In [1]:
from openai import OpenAI
import json

In [2]:
client = OpenAI(api_key="your_own_api_key")

In [3]:
MAX_ITERATIONS = 5
agent_memory=[]

In [4]:
def generate_code(task, feedback=None):
    system_prompt = (
        "You are a Python code-writing agent. "
        "Generate clean, correct, runnable Python code. "
        "Include comments where appropriate. "
        "Do not include explanations outside the code block. "
        "Apply feedback carefully if provided."
    )
    
    user_prompt = f"TASK:\n{task}\n"
    
    if feedback:
        user_prompt += f"\nFEEDBACK:\n{feedback}"
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    return response.choices[0].message.content.strip()

In [14]:
def clean_code_block(code):
    # Remove markdown code fences if present
    if code.startswith("```"):
        code = code.strip("`")
        lines = code.split("\n")
        # Remove first line if it says python
        if lines[0].lower().startswith("python"):
            lines = lines[1:]
        code = "\n".join(lines)
    return code.strip()


def check_syntax(code):
    code = clean_code_block(code)
    
    try:
        compile(code, "<string>", "exec")
        return True, None
    except Exception as e:
        return False, str(e)

In [15]:
def evaluate_code(code):
    syntax_ok, syntax_error = check_syntax(code)
    
    system_prompt = (
        "You are a strict Python code reviewer. "
        "Evaluate the following code for:\n"
        "- Logical correctness\n"
        "- Clean structure\n"
        "- Proper comments\n"
        "- Simplicity\n\n"
        "Return ONLY valid JSON:\n"
        "{\n"
        '  "logic_ok": true/false,\n'
        '  "clean_structure": true/false,\n'
        '  "well_commented": true/false,\n'
        '  "issues": [string],\n'
        '  "suggestions": [string]\n'
        "}"
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": code}
        ]
    )
    
    eval_data = json.loads(response.choices[0].message.content)
    
    eval_data["syntax_ok"] = syntax_ok
    eval_data["syntax_error"] = syntax_error
    
    eval_data["passed"] = (
        syntax_ok
        and eval_data["logic_ok"]
        and eval_data["clean_structure"]
        and eval_data["well_commented"]
    )
    
    return eval_data

In [16]:
def run_code_agent(task):
    feedback = None
    best_code = None
    best_score = -1
    
    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n=== Iteration {iteration} ===")
        
        code = generate_code(task, feedback)
        evaluation = evaluate_code(code)
        
        score = sum([
            evaluation["syntax_ok"],
            evaluation["logic_ok"],
            evaluation["clean_structure"],
            evaluation["well_commented"]
        ])
        
        agent_memory.append({
            "iteration": iteration,
            "code": code,
            "evaluation": evaluation,
            "feedback": feedback,
            "score": score
        })
        
        print("Syntax OK:", evaluation["syntax_ok"])
        print("Checks passed:", score, "/ 4")
        
        if evaluation["passed"]:
            print(" Code meets requirements. Stopping.")
            return code
        
        if score > best_score:
            best_score = score
            best_code = code
        
        if not evaluation["syntax_ok"]:
            feedback = f"Fix syntax error: {evaluation['syntax_error']}"
        else:
            feedback = "\n".join(evaluation["suggestions"])
        
        print(" Refining code...")
    
    print(" Max iterations reached. Returning best attempt.")
    return best_code

In [17]:
task = "Write a Python function that checks if a number is prime."
final_code = run_code_agent(task)
print("\nFINAL CODE:\n")
print(final_code)


=== Iteration 1 ===
Syntax OK: True
Checks passed: 4 / 4
 Code meets requirements. Stopping.

FINAL CODE:

```python
def is_prime(n):
    """Check if a number is prime."""
    if n <= 1:
        return False  # 0 and 1 are not prime numbers
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False  # Found a divisor, so it's not prime
    return True  # No divisors found, it's a prime number

# Example usage:
number = 29
if is_prime(number):
    print(f"{number} is a prime number.")
else:
    print(f"{number} is not a prime number.")
```
